# 03 · Join raster climate to vector boundaries and non-spatial population data

**joining geospatial with non-geospatial data.** Three very different shapes of data meet on one key, the **H3 cell**:

| Input | Shape | How it gets to H3 |
|---|---|---|
| BARPA-R climate (from 02) | raster | `gbx_rst_h3_rastertogridavg` (GeoBrix) |
| ABS SA2 boundaries | vector polygons | `shapefile_gbx` → `GEOMETRY` → `h3_coverash3` / `h3_longlatash3` (native Databricks) |
| ABS Estimated Resident Population | **plain Excel table**, no geometry | joins to SA2 on `SA2_CODE21` |

Outputs are gold tables for the app, Genie, and ArcGIS (via ODBC or Data Pipelines), plus a **reusable UC SQL function**.

In [ ]:
%pip install -q "geobrix[light_env6] @ https://github.com/databrickslabs/geobrix/releases/download/v0.5.2/geobrix-0.5.2-py3-none-any.whl" openpyxl

In [ ]:
%restart_python

In [ ]:
%run ./00_config

In [ ]:
from pyspark.sql import functions as F
from databricks.labs.gbx.ds.register import register
register(spark)

## 1 · Vector: ABS SA2 boundaries → native `GEOMETRY`
GeoBrix reads the zipped shapefile straight from the Volume. GDA2020 (EPSG:7844) and WGS84 differ by under 2 m, so for H3 we treat the coordinates as lon/lat.

In [ ]:
sa2_raw = spark.read.format("shapefile_gbx").load(f"{VOL}/abs/SA2_2021_AUST_SHP_GDA2020.shp.zip")
sa2 = (sa2_raw.where("geom_0 IS NOT NULL")                        # drops 'No usual address' / migratory SA2s
       .select(F.col("SA2_CODE21").alias("sa2_code"), F.col("SA2_NAME21").alias("sa2_name"),
               F.col("SA4_NAME21").alias("sa4_name"), F.col("STE_NAME21").alias("state"),
               F.col("AREASQKM21").alias("area_km2"),
               F.expr("st_geomfromwkb(geom_0, 4326)").alias("geom")))
sa2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.silver_sa2_boundaries")
sa2 = spark.table(f"{FQ}.silver_sa2_boundaries")
print(f"{sa2.count():,} SA2 polygons with geometry")
display(sa2.select("sa2_code", "sa2_name", "state", "area_km2", F.expr("st_geometrytype(geom)").alias("type"),
                   F.expr("st_npoints(geom)").alias("vertices")).limit(5))

## 2 · Non-spatial: ABS population (Excel) → Delta

In [ ]:
import pandas as pd
xl = pd.read_excel(f"{VOL}/abs/erp_sa2_2001_2025.xlsx", sheet_name="Table 1", header=None, skiprows=6)
xl = xl.iloc[:, [0, 1, 8, 9, 30, 34]]                                # S/T code, S/T name, SA2 code, SA2 name, ERP 2021, ERP 2025
xl.columns = ["ste_code", "ste_name", "sa2_code", "sa2_name", "erp_2021", "erp_2025"]
xl = xl[pd.to_numeric(xl["sa2_code"], errors="coerce").notna()].copy()
xl["sa2_code"] = xl["sa2_code"].astype("int64").astype(str)
for c in ["erp_2021", "erp_2025"]:
    xl[c] = pd.to_numeric(xl[c], errors="coerce").fillna(0).astype("int64")
pop = spark.createDataFrame(xl)
pop.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.silver_sa2_population")
print(f"{len(xl):,} SA2 rows · Australia ERP 2025 = {xl.erp_2025.sum():,}")

## 3 · Put everything on the H3 grid
* **Land mask:** an H3 cell is "land" if any SA2 polygon covers it (`h3_coverash3`). That removes ocean pixels from the climate grid.
* **Population:** each SA2's population is placed at its centroid's H3 cell (`h3_longlatash3`).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.silver_h3_land AS
WITH cover AS (
  SELECT sa2_code, state, area_km2,
         explode(h3_coverash3(st_aswkb(st_simplify(geom, 0.01)), {H3_RES})) AS cell_id
  FROM {FQ}.silver_sa2_boundaries
)
SELECT cell_id,
       max_by(state, area_km2)                        AS state,
       count(DISTINCT sa2_code)                       AS n_sa2,
       st_y(st_geomfromwkt(h3_centeraswkt(cell_id)))  AS lat,
       st_x(st_geomfromwkt(h3_centeraswkt(cell_id)))  AS lon
FROM cover GROUP BY cell_id
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.silver_sa2_cell AS
SELECT b.sa2_code, b.sa2_name, b.sa4_name, b.state, b.area_km2,
       h3_longlatash3(st_x(st_centroid(b.geom)), st_y(st_centroid(b.geom)), {H3_RES}) AS cell_id,
       coalesce(p.erp_2025, 0) AS population, coalesce(p.erp_2021, 0) AS population_2021
FROM {FQ}.silver_sa2_boundaries b LEFT JOIN {FQ}.silver_sa2_population p USING (sa2_code)
""")
display(spark.sql(f"SELECT count(*) land_cells, count(DISTINCT state) states FROM {FQ}.silver_h3_land"))

## 4 · Multidimensional reduction in SQL: cell × scenario × decade metrics
From the H3 cell × month table we derive, per cell:
* `tasmax_ann`: mean daily maximum temperature (°C)
* `tasmax_djf`: summer (Dec-Feb) mean daily maximum (°C)
* `hot_months`: average number of months a year with mean daily max ≥ 35 °C
* `pr_ann_mm`: annual rainfall (mm)

Then each future is **normalised against the 2001-2010 baseline** as a change (Δ °C, Δ months, % rainfall).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.gold_h3_climate
COMMENT 'Per H3 res-{H3_RES} land cell climate metrics for each CMIP6 scenario and decade (BARPA-R ACCESS-CM2), with change vs 2001-2010 baseline'
AS
WITH m AS (
  SELECT cell_id, scenario, decade,
         avg(CASE WHEN variable = 'tasmax' THEN value END)                                         AS tasmax_ann,
         avg(CASE WHEN variable = 'tasmax' AND month IN (12, 1, 2) THEN value END)                 AS tasmax_djf,
         count(CASE WHEN variable = 'tasmax' AND value >= {HOT_MONTH_C} THEN 1 END) / 10.0          AS hot_months,
         avg(CASE WHEN variable = 'pr' THEN value END) * 365.25                                     AS pr_ann_mm
  FROM {FQ}.silver_h3_monthly
  WHERE cell_id IN (SELECT cell_id FROM {FQ}.silver_h3_land)
  GROUP BY ALL
),
b AS (SELECT * FROM m WHERE scenario = '{BASELINE[0]}')
SELECT f.cell_id, l.state, l.lat, l.lon,
       f.scenario, f.decade,
       CASE f.decade WHEN '200101-201012' THEN '2001-2010' WHEN '204101-205012' THEN '2041-2050' ELSE '2091-2100' END AS period,
       round(f.tasmax_ann, 2) AS tasmax_ann, round(f.tasmax_djf, 2) AS tasmax_djf,
       round(f.hot_months, 2) AS hot_months, round(f.pr_ann_mm, 1) AS pr_ann_mm,
       round(f.tasmax_ann - b.tasmax_ann, 2)               AS d_tasmax_ann,
       round(f.tasmax_djf - b.tasmax_djf, 2)               AS d_tasmax_djf,
       round(f.hot_months - b.hot_months, 2)               AS d_hot_months,
       round((f.pr_ann_mm - b.pr_ann_mm) / b.pr_ann_mm * 100, 1) AS pct_pr,
       coalesce(p.population, 0)                           AS population
FROM m f
JOIN b USING (cell_id)
JOIN {FQ}.silver_h3_land l USING (cell_id)
LEFT JOIN (SELECT cell_id, sum(population) population FROM {FQ}.silver_sa2_cell GROUP BY cell_id) p USING (cell_id)
""")
display(spark.sql(f"""SELECT scenario, period, count(*) cells, round(avg(tasmax_ann),2) tasmax_ann,
                    round(avg(d_tasmax_ann),2) d_tasmax, round(avg(hot_months),2) hot_months, round(avg(pct_pr),1) pct_pr
                    FROM {FQ}.gold_h3_climate GROUP BY ALL ORDER BY period, scenario"""))

## 5 · Exposure: who lives where it gets hotter? (SA2 + population + climate)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.gold_sa2_exposure
COMMENT 'ABS SA2 areas with 2025 population joined to projected summer heat (BARPA-R) by scenario and decade'
AS
SELECT s.sa2_code, s.sa2_name, s.sa4_name, s.state, s.population, s.area_km2,
       c.scenario, c.period, c.tasmax_djf, c.d_tasmax_djf, c.hot_months, c.d_hot_months, c.pct_pr
FROM {FQ}.silver_sa2_cell s JOIN {FQ}.gold_h3_climate c USING (cell_id)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.gold_exposure_summary
COMMENT 'Population-weighted climate change and heat exposure by state, scenario and decade'
AS
SELECT coalesce(state, 'Australia') AS state, scenario, period,
       sum(population)                                                          AS population,
       round(sum(population * d_tasmax_djf) / sum(population), 2)              AS pop_weighted_d_tasmax_djf,
       sum(CASE WHEN hot_months >= 1 THEN population ELSE 0 END)               AS pop_with_hot_months,
       sum(CASE WHEN tasmax_djf >= 35 THEN population ELSE 0 END)              AS pop_summer_max_35c
FROM {FQ}.gold_sa2_exposure
GROUP BY GROUPING SETS ((state, scenario, period), (scenario, period))
""")
display(spark.sql(f"SELECT * FROM {FQ}.gold_exposure_summary WHERE state = 'Australia' ORDER BY period, scenario"))

## 6 · State outlines for the 3D app (simplified GeoJSON)

In [ ]:
ste = spark.read.format("shapefile_gbx").load(f"{VOL}/abs/STE_2021_AUST_SHP_GDA2020.shp.zip")
(ste.where("geom_0 IS NOT NULL AND STE_CODE21 NOT IN ('9', 'Z')")
    .select(F.col("STE_NAME21").alias("state"),
            F.expr("st_asgeojson(st_simplify(st_geomfromwkb(geom_0, 4326), 0.05))").alias("geojson"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.gold_state_outlines"))
display(spark.sql(f"SELECT state, length(geojson) AS geojson_chars FROM {FQ}.gold_state_outlines"))

## 7 · A reusable, governed method: a UC SQL table function
Write once, use everywhere: SQL editor, notebooks, Genie, the app, BI tools, and ArcGIS over ODBC all call the same governed logic.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.heat_exposure(p_scenario STRING, p_period STRING, p_min_warming DOUBLE)
RETURNS TABLE (state STRING, sa2_count BIGINT, population BIGINT, avg_d_tasmax_djf DOUBLE)
COMMENT 'People living in SA2s where summer max temperature rises by at least p_min_warming degC (BARPA-R, vs 2001-2010)'
RETURN
  SELECT state, count(*) AS sa2_count, sum(population) AS population, round(avg(d_tasmax_djf), 2) AS avg_d_tasmax_djf
  FROM {FQ}.gold_sa2_exposure
  WHERE scenario = p_scenario AND period = p_period AND d_tasmax_djf >= p_min_warming
  GROUP BY state
""")
display(spark.sql(f"SELECT * FROM {FQ}.heat_exposure('ssp370', '2091-2100', 3.0) ORDER BY population DESC"))